# EXPERIMENT 3B – K-Nearest Neighbours (KNN) on Social Network Ads

**Aim:** To implement the K-Nearest Neighbours algorithm on the Social Network Ads dataset and calculate the confusion matrix, accuracy, error rate, precision and recall.

**Dataset:** [Social Network Ads – Kaggle](https://www.kaggle.com/datasets/rakeshrau/social-network-ads)

**Target:** `Purchased` (0 = No, 1 = Yes).

**Features used:** `Age` and `EstimatedSalary`. `User ID` is only an identifier, so it is not used for prediction. This beginner-friendly implementation also leaves out `Gender`.

## 1. Import required libraries

Pandas loads the CSV file. Scikit-learn provides the train/test split, feature scaling, KNN model and evaluation metrics.

In [ ]:
# Import libraries for data handling and display
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Split data and scale numerical features
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# K-Nearest Neighbours classifier
from sklearn.neighbors import KNeighborsClassifier

# Evaluation metrics
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    classification_report
)

## 2. Load the dataset

**Before running:** Download `Social_Network_Ads.csv` from the Kaggle dataset link above and place it in the same folder as this notebook. If your file is in another folder, change the path in `pd.read_csv()`.

The alternative direct CSV URL below is a public mirror of the commonly used dataset; use the Kaggle download if your university requires the original source.

In [ ]:
# Read the CSV file from the notebook's current folder
# Change the path if the CSV file is stored in another folder.
try:
    dataset = pd.read_csv("Social_Network_Ads.csv")
except FileNotFoundError:
    # Public mirror fallback; internet access is required for this line.
    csv_url = "https://huggingface.co/datasets/Rodrigopiva/Social_Network_Ads.csv/resolve/main/Social_Network_Ads.csv"
    dataset = pd.read_csv(csv_url)

# Display the first five rows
dataset.head()

## 3. Explore the dataset
Check the number of rows, column names, data types and missing values before training.

In [ ]:
# Display the shape (rows, columns) and column information
print("Dataset shape:", dataset.shape)
print("\nColumn information:")
dataset.info()

# Count missing values in every column
print("\nMissing values:")
print(dataset.isnull().sum())

# Display basic statistical information for numerical columns
print("\nStatistical summary:")
display(dataset.describe())

## 4. Select input features and target
We use Age and EstimatedSalary as input features (`X`) and Purchased as the target (`y`). The user ID is not useful for learning customer behaviour.

In [ ]:
# Select the two numerical input columns
X = dataset[["Age", "EstimatedSalary"]]

# Select the column that the model must predict
y = dataset["Purchased"]

print("Input features:")
display(X.head())

print("\nTarget values:")
print(y.head())

## 5. Split data into training and testing sets
80% of records are used to train the model and 20% to test it. Stratification preserves the proportion of purchased/not-purchased records.

In [ ]:
# Split the dataset into training and testing portions
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

## 6. Feature scaling
KNN uses distance to find nearby records. Salary values are much larger than age values, so standardization prevents salary from dominating the distance calculation.

In [ ]:
# Create a StandardScaler object
scaler = StandardScaler()

# Fit scaler on training data and transform it
X_train_scaled = scaler.fit_transform(X_train)

# Transform test data using the same training-set scaling values
X_test_scaled = scaler.transform(X_test)

## 7. Create and train the KNN classifier
Here, `n_neighbors=5` means the model looks at the five nearest training records. The most common class among those neighbours becomes the prediction.

In [ ]:
# Create the KNN classifier with 5 nearest neighbours
knn_model = KNeighborsClassifier(n_neighbors=5)

# Train the classifier using scaled training features and target labels
knn_model.fit(X_train_scaled, y_train)

print("KNN model training completed.")

## 8. Predict the test set results
Use the trained model to predict whether each test-set user purchased the product.

In [ ]:
# Predict purchase outcomes for the test records
y_pred = knn_model.predict(X_test_scaled)

# Compare a few actual values with the predicted values
comparison = pd.DataFrame({
    "Actual": y_test.to_numpy()[:10],
    "Predicted": y_pred[:10]
})
display(comparison)

## 9. Calculate confusion matrix and performance measures
For binary classification, the confusion matrix is arranged as `[[TN, FP], [FN, TP]]` when labels are `[0, 1]`.

- **Accuracy** = (TP + TN) / (TP + TN + FP + FN)
- **Error rate** = (FP + FN) / Total
- **Precision** = TP / (TP + FP)
- **Recall** = TP / (TP + FN)

In [ ]:
# Calculate confusion matrix; label 0 = not purchased, 1 = purchased
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

print("Confusion Matrix:")
print(cm)
print("\nTrue Negative (TN):", tn)
print("False Positive (FP):", fp)
print("False Negative (FN):", fn)
print("True Positive (TP):", tp)

# Calculate the requested performance measures
accuracy = accuracy_score(y_test, y_pred)
error_rate = 1 - accuracy
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)

print("\nAccuracy:   ", round(accuracy, 4))
print("Error Rate: ", round(error_rate, 4))
print("Precision:  ", round(precision, 4))
print("Recall:     ", round(recall, 4))

## 10. Display the confusion matrix as a heatmap
The heatmap makes it easier to see correct and incorrect predictions for both classes.

In [ ]:
# Plot the confusion matrix using Matplotlib
plt.figure(figsize=(5, 4))
plt.imshow(cm, interpolation="nearest", cmap="Blues")
plt.title("KNN Confusion Matrix")
plt.colorbar()

# Add axis labels and class names
class_names = ["Not Purchased (0)", "Purchased (1)"]
tick_marks = np.arange(len(class_names))
plt.xticks(tick_marks, class_names, rotation=20)
plt.yticks(tick_marks, class_names)
plt.xlabel("Predicted Label")
plt.ylabel("Actual Label")

# Write each cell's count inside the heatmap
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, str(cm[i, j]), ha="center", va="center")

plt.tight_layout()
plt.show()

## 11. Classification report
This report gives precision, recall and F1-score for each class (0 and 1).

In [ ]:
# Print detailed classification metrics for both classes
print(classification_report(
    y_test,
    y_pred,
    target_names=["Not Purchased", "Purchased"],
    zero_division=0
))

## 12. Conclusion

The K-Nearest Neighbours algorithm was successfully implemented on the Social Network Ads dataset. The model predicts whether a user purchases a product based on age and estimated salary. Its performance was evaluated using the confusion matrix, accuracy, error rate, precision and recall.

**Result:** The numerical metric values are displayed when the notebook is executed. They depend on the dataset and the train/test split.